# LeetCode Hot 100 - Day 12

## 今日主题：堆与链表困难题

今天的主角是**堆（优先队列）**，另外两题是链表的困难题：

1. 数组中的第 K 个最大元素：堆的第一种用法——用大小为 k 的小顶堆选出前 k 大。
2. 前 K 个高频元素：先统计频率，再用堆选前 k 个。
3. K 个一组翻转链表：把 Day 05 的整条反转，改成一段一段反转。
4. 合并 K 个升序链表：堆在链表上的应用，加练题。

堆是面试高频考点，Python 里就是 `heapq` 这个内置库。今天第一次用，会从零讲清楚。


## 今天怎么学

1. 先学第一题，把 `heapq` 的三个基本操作用熟：`heappush`、`heappop`、`heap[0]`。
2. 第二题是同一套模板加一个字典计数，写得会很快。
3. 第三题是今天的难点，动手前先在纸上画一遍“三个指针怎么走”。
4. 第四题有余力再做。

最低目标：独立写出第一题和第二题，能说清“为什么求第 k 大要用小顶堆，而不是大顶堆”。


## 今日题单

1. 数组中的第 K 个最大元素（LeetCode 215，中等，必做）
2. 前 K 个高频元素（LeetCode 347，中等，必做）
3. K 个一组翻转链表（LeetCode 25，困难，必做）
4. 合并 K 个升序链表（LeetCode 23，困难，加练）


## 昨日复习

先用 `day11_practice.ipynb` 不看答案重写：

1. 两两交换链表中的节点：重点是三条指针的顺序，以及 `pre` 移到哪里。
2. LRU 缓存：重点是三个小方法 `add_to_head`、`remove_node`、`move_to_head`。

口述：LRU 为什么必须用双向链表？


## 今天第一件要做的事：认识 heapq

前面所有题用的容器都是列表、字典、集合。今天要加一个新工具：**堆**。

**堆是什么？** 一种“随时能拿到当前最小值”的容器。Python 里没有单独的堆类型，用列表加 `heapq` 这个模块来实现，而且**默认是最小堆**：

- `heapq.heappush(heap, x)`：把 x 放进堆，自动调整位置。
- `heapq.heappop(heap)`：弹出并返回**最小的**元素。
- `heap[0]`：看一眼最小值，不弹出。
- `heapq.heapify(列表)`：把一个普通列表原地变成堆。

每次 push 和 pop 都是 O(log n)，比“每次都排序”快得多。

先跑一遍看清楚它的行为。


In [ ]:
import heapq

heap = []
heapq.heappush(heap, 5)
heapq.heappush(heap, 1)
heapq.heappush(heap, 3)
print("堆里现在是：", heap)
print("看一眼最小值（不弹出）：", heap[0])

print("弹出最小值：", heapq.heappop(heap))
print("再弹一个：", heapq.heappop(heap))
print("剩下的：", heap)

# 注意：直接打印堆，看到的是“列表的存储顺序”，不是完全排好序的
numbers = [7, 2, 9, 4]
heapq.heapify(numbers)
print("heapify 之后：", numbers, "，最小值是", numbers[0])

# 想要大顶堆？Python 没有，习惯做法是存负数
max_heap = []
for value in [5, 1, 3]:
    heapq.heappush(max_heap, -value)
print("负数堆的堆顶是：", -max_heap[0])


## 题目 1 做题前先补：为什么是“大小为 k 的小顶堆”

题目要第 k 个**最大**的元素，直觉上想用大顶堆。但请这样想：

- 如果维护一个容量为 `k` 的容器，装“目前见过的、最大的 k 个元素”；
- 容器满了还要塞新元素时，先比较一下：**容器里最小的那个**和新元素谁大，谁大留谁。

所以这个容器需要随时能拿到“自己内部的最小值”，也就是**小顶堆**。

结论：**求前 k 大，用小顶堆，堆顶就是答案；求前 k 小，用大顶堆。** 这一点面试官很喜欢问。

流程（以 `nums = [3, 2, 1, 5, 6, 4]`，`k = 2` 为例）：

1. 依次把数字放进堆。
2. 堆的大小超过 2 时，弹掉堆顶（最小的那个）。
3. 全部走完后，堆里就是最大的 2 个，堆顶就是第 2 大。


In [ ]:
import heapq

nums = [3, 2, 1, 5, 6, 4]
k = 2
heap = []

for num in nums:
    heapq.heappush(heap, num)
    if len(heap) > k:
        removed = heapq.heappop(heap)
        print("放入", num, "后超容量，弹掉", removed, "，堆变成", heap)
    else:
        print("放入", num, "，堆变成", heap)

print("最终的堆：", heap)
print("第", k, "大的元素是堆顶：", heap[0])


# 题目 1：数组中的第 K 个最大元素

LeetCode 215. Kth Largest Element in an Array

## 题目描述（改写版）

给你一个整数数组 `nums` 和一个整数 `k`，返回数组中**第 k 个最大**的元素。

注意“第 k 个最大”的意思是：把数组从大到小排序后，位于第 k 个位置的元素。重复的数字算多个，不是“第 k 个不同的元素”。

## 输入

- `nums`：整数数组，长度 1 到 100000，元素范围 -10000 到 10000。
- `k`：1 到 `nums` 的长度。

## 输出

返回第 k 大的元素（是一个数，不是下标）。

## 示例

示例 1：`nums = [3, 2, 1, 5, 6, 4]`，`k = 2`，从大到小排是 `[6, 5, 4, 3, 2, 1]`，第 2 个是 5。

示例 2：`nums = [3, 2, 3, 1, 2, 4, 5, 5, 6]`，`k = 4`，从大到小排是 `[6, 5, 5, 4, 3, 3, 2, 2, 1]`，第 4 个是 4。

## 易漏细节

- 允许重复元素，`k = 1` 时答案是最大值。
- 不要把“第 k 大”理解成“第 k 个不同值”。


## 解法名称

**小顶堆维护前 K 大（Min-Heap of Size K）**。

## 暴力思路

直接排序，取倒数第 k 个：`sorted(nums)[-k]`。时间 O(n log n)，能过题但没体现堆的价值，面试官会追问“能不能更快”。

## 优化思路

维护一个大小不超过 k 的小顶堆：

- 遍历每个数字，`heappush` 进堆。
- 堆大小超过 k 时，`heappop` 弹掉堆顶，也就是弹掉“目前这 k+1 个里最小的”。
- 遍历结束后，堆里保留的就是最大的 k 个元素，堆顶是它们当中最小的，也就是整个数组的第 k 大。

复杂度：每个元素最多进堆、出堆各一次，每次 O(log k)，所以时间 O(n log k)，比排序的 O(n log n) 好；空间 O(k)。

如果面试官追问更快的方法，可以提“快速选择（Quick Select）平均 O(n)”，但堆的写法最稳、最容易解释。


## 你来写：第 K 个最大元素

要求：

- 用 `heapq` 的小顶堆写，堆的大小不超过 k。
- 写完用 `[3,2,1,5,6,4], k=2` 和只有 1 个元素的数组各跑一遍。

先在心里回答：为什么求第 k 大用的是小顶堆，而求第 k 小要用大顶堆？


In [ ]:
import heapq

# 题目：数组中的第 K 个最大元素
# 解法：小顶堆维护前 K 大（Min-Heap of Size K）
# 输入：整数数组 nums（长度 1 到 100000），整数 k（1 到 len(nums)）。
# 目标：找出从大到小排序后第 k 个位置的元素。
# 输出：返回这个元素的值。
# 注意：允许重复元素，重复算多个；堆的大小超过 k 时弹出堆顶。


def find_kth_largest(nums, k):
    # 在这里写你的代码
    pass


print(find_kth_largest([3, 2, 1, 5, 6, 4], 2))


In [ ]:
print(find_kth_largest([3, 2, 1, 5, 6, 4], 2))                        # 期望 5
print(find_kth_largest([3, 2, 3, 1, 2, 4, 5, 5, 6], 4))               # 期望 4
print(find_kth_largest([1], 1))                                       # 期望 1
print(find_kth_largest([-1, -2, -3], 1))                              # 期望 -1
print(find_kth_largest([2, 2, 2], 2))                                 # 期望 2


## 参考答案：第 K 个最大元素

先自己写完再看。

```python
import heapq


def find_kth_largest_answer(nums, k):
    heap = []
    for num in nums:
        heapq.heappush(heap, num)
        if len(heap) > k:
            heapq.heappop(heap)
    return heap[0]
```

面试表达：

我用一个大小为 k 的小顶堆。遍历数组，每个元素先进堆；堆里超过 k 个时，弹掉堆顶，也就是弹掉当前这 k+1 个里最小的那个。遍历结束后，堆里就是最大的 k 个元素，堆顶就是第 k 大。时间 O(n log k)，空间 O(k)。如果数组特别大、k 特别小，这个做法比完整排序更省时间。


In [ ]:
import heapq


def find_kth_largest_answer(nums, k):
    heap = []
    for num in nums:
        heapq.heappush(heap, num)
        if len(heap) > k:
            heapq.heappop(heap)
    return heap[0]


print(find_kth_largest_answer([3, 2, 1, 5, 6, 4], 2))      # 5
print(find_kth_largest_answer([3, 2, 3, 1, 2, 4, 5, 5, 6], 4))   # 4
print(find_kth_largest_answer([1], 1))                     # 1
print(find_kth_largest_answer([-1, -2, -3], 1))            # -1


## 题目 2 做题前先补：把元组放进堆

上一题的堆里放的是整数。这一题要放两个信息：**频率**和**数字**。

做法是把它们打包成元组 `(频率, 数字)` 放进堆。元组比较大小的时候，先比第一个元素；第一个元素相同时，再比第二个。

所以 `(3, 5)` 比 `(2, 9)` 大，因为 3 > 2，跟数字本身无关。这正是我们要的：堆顶是频率最小的那个。

```python
heapq.heappush(heap, (2, 9))
heapq.heappush(heap, (3, 5))
heapq.heappop(heap)   # 弹出 (2, 9)，因为它的频率更小
```

小结：**要让堆按什么排序，就把那个东西放在元组的第一个位置。**

另外补充一点：`(频率, 数字)` 里第二个元素是整数，比较时不会出问题。如果第二个元素是链表节点这种不能比较大小、又可能相等的东西，就得再加一个不会重复的数字做“裁判”（第四题会遇到）。


In [ ]:
import heapq

heap = []
heapq.heappush(heap, (2, 9))    # 数字 9 出现 2 次
heapq.heappush(heap, (3, 5))    # 数字 5 出现 3 次
heapq.heappush(heap, (1, 7))    # 数字 7 出现 1 次

print("堆：", heap)
print("堆顶（频率最小的）：", heap[0])
print("弹出：", heapq.heappop(heap))


# 题目 2：前 K 个高频元素

LeetCode 347. Top K Frequent Elements

## 题目描述（改写版）

给你一个整数数组 `nums` 和一个整数 `k`，请返回其中**出现频率最高的 k 个元素**。

返回顺序不限。

## 输入

- `nums`：整数数组，长度 1 到 100000。
- `k`：1 到“不同元素的个数”。

## 输出

返回一个列表，里面是频率最高的 k 个元素（顺序任意）。

## 示例

示例 1：`nums = [1, 1, 1, 2, 2, 3]`，`k = 2`。1 出现 3 次，2 出现 2 次，3 出现 1 次，所以返回 `[1, 2]`。

示例 2：`nums = [1]`，`k = 1`，返回 `[1]`。

## 易漏细节

- 要先统计频率，再选前 k 个，不能直接对原数组操作。
- 频率相同的元素，返回哪个都算对，不要写死顺序。


## 解法名称

**哈希计数 + 小顶堆（Hash Count + Min-Heap）**。

## 暴力思路

统计完频率后，把所有元素按频率排序，取前 k 个。时间 O(m log m)（m 是不同元素个数），能过但不够快。

## 优化思路

第一步：用字典统计每个数字出现的次数（Day 01 就学过）。

第二步：和上一题一模一样的思路——维护一个**大小为 k 的小顶堆**，只不过堆里放的是 `(频率, 数字)`。

- 每个键值对 `(频率, 数字)` 进堆。
- 堆大小超过 k 就弹出堆顶，也就是弹出频率最小的。
- 最后堆里剩下的就是频率最高的 k 个。

时间 O(n + m log k)，空间 O(m)。其中 n 是数组长度，m 是不同元素个数。

如果面试官问“能不能做到 O(n)”，可以答桶排序：把频率当作下标放进桶里（频率范围是 1 到 n），从后往前取 k 个，时间是 O(n)。


## 你来写：前 K 个高频元素

要求：

- 第一步用字典计数，第二步用大小为 k 的小顶堆。
- 注意堆里存的是元组 `(频率, 数字)`。
- 最后把堆里的数字取出来放进结果列表返回。

先在心里回答：为什么堆里要把频率放在元组的第一位？


In [ ]:
import heapq

# 题目：前 K 个高频元素
# 解法：哈希计数 + 小顶堆（Hash Count + Min-Heap）
# 输入：整数数组 nums（长度 1 到 100000），整数 k（1 到 不同元素个数）。
# 目标：找出出现频率最高的 k 个元素。
# 输出：返回这 k 个元素组成的列表，顺序不限。
# 注意：先用字典计数，再用大小为 k 的小顶堆；堆里存 (频率, 数字)，频率放在第一位。


def top_k_frequent(nums, k):
    # 在这里写你的代码
    pass


print(top_k_frequent([1, 1, 1, 2, 2, 3], 2))


In [ ]:
result = top_k_frequent([1, 1, 1, 2, 2, 3], 2)
if result is None:
    print("top_k_frequent 还没有返回结果，先把上面的函数写完再运行这一格。")
else:
    print(sorted(result))                                   # 期望 [1, 2]
    print(sorted(top_k_frequent([1], 1)))                    # 期望 [1]
    print(sorted(top_k_frequent([4, 4, 4, 4], 1)))           # 期望 [4]
    print(sorted(top_k_frequent([1, 2, 1, 2, 3], 2)))        # 期望 [1, 2]
    print(sorted(top_k_frequent([5, 3, 5, 3, 5, 3, 7], 1)))  # 期望 [3] 或 [5]，频率都是 3


## 参考答案：前 K 个高频元素

```python
import heapq


def top_k_frequent_answer(nums, k):
    count = {}
    for num in nums:
        if num in count:
            count[num] = count[num] + 1
        else:
            count[num] = 1

    heap = []
    for num in count:
        heapq.heappush(heap, (count[num], num))
        if len(heap) > k:
            heapq.heappop(heap)

    result = []
    for item in heap:
        result.append(item[1])
    return result
```

面试表达：

我先用字典统计每个元素出现的次数，然后维护一个大小为 k 的小顶堆，堆里存 `(频率, 数字)` 元组。因为元组比较时先比频率，所以堆顶就是当前频率最小的那个；堆超过 k 个就弹掉堆顶。遍历完所有元素后，堆里剩下的就是频率最高的 k 个。时间 O(n + m log k)，空间 O(m)。如果想要严格的 O(n)，可以改成桶排序，用频率当下标。


In [ ]:
import heapq


def top_k_frequent_answer(nums, k):
    count = {}
    for num in nums:
        if num in count:
            count[num] = count[num] + 1
        else:
            count[num] = 1

    heap = []
    for num in count:
        heapq.heappush(heap, (count[num], num))
        if len(heap) > k:
            heapq.heappop(heap)

    result = []
    for item in heap:
        result.append(item[1])
    return result


print(sorted(top_k_frequent_answer([1, 1, 1, 2, 2, 3], 2)))     # [1, 2]
print(sorted(top_k_frequent_answer([1], 1)))                    # [1]
print(sorted(top_k_frequent_answer([1, 2, 1, 2, 3], 2)))        # [1, 2]
print(sorted(top_k_frequent_answer([5, 3, 5, 3, 5, 3, 7], 1)))  # [3] 或 [5]


## 题目 3 做题前先补：从“整条反转”到“一段一段反转”

Day 05 的反转链表，核心就一句话：一路走一路把指针掉头。

```python
prev = None
current = head
while current is not None:
    nxt = current.next
    current.next = prev
    prev = current
    current = nxt
```

今天要把它升级：**每 k 个节点为一段，段内反转，段与段保持原来的先后顺序**。

升级需要多管四个变量，先认清楚：

| 变量 | 含义 |
| --- | --- |
| `group_prev` | 上一段的最后一个节点。它也是**本段的前驱**，本段反转后要由它指向本段的新头 |
| `kth` | 本段的第 k 个节点。反转后它变成本段的**新头** |
| `group_next` | 本段后面那一段的第一个节点（可能为 None） |
| `old_start` | 本段原来的第一个节点。反转后它变成本段的**尾巴**，下一轮由它当 `group_prev` |

反转一段时有个小技巧：让 `prev` 从 `group_next` 开始，而不是从 `None` 开始。这样反转结束后，原来这段的第一个节点自然就指向了下一段的开头，不用再单独补一次连接。

先手工做一遍，看清 `prev` 从 `group_next` 出发的效果。


In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


def build_list(values):
    """根据普通列表从头创建一条无环单链表。"""
    dummy = ListNode()
    tail = dummy
    for value in values:
        tail.next = ListNode(value)
        tail = tail.next
    return dummy.next


def list_values(head, limit=30):
    """把无环链表转成普通列表，方便观察结果。"""
    values = []
    current = head
    while current is not None and len(values) < limit:
        values.append(current.val)
        current = current.next
    return values


# 手工反转 1 -> 2 -> 3 -> 4 中的前两个节点，后面还留着 3 -> 4
head = build_list([1, 2, 3, 4])
group_prev = ListNode()          # 假装它是 dummy
group_prev.next = head
kth = head.next                  # 本段第 2 个节点
group_next = kth.next            # 下一段的开头：3
print("本段：", group_prev.next.val, "->", kth.val, "，下一段开头是：", group_next.val)

prev = group_next                # 关键：prev 从下一段的开头出发
current = group_prev.next
while current is not group_next:
    nxt = current.next
    current.next = prev
    prev = current
    current = nxt

old_start = group_prev.next
group_prev.next = kth            # 前驱指向本段新头
print("反转后：", list_values(group_prev.next))
print("本段的新尾巴是：", old_start.val, "，它现在指向：", old_start.next.val)


# 题目 3：K 个一组翻转链表

LeetCode 25. Reverse Nodes in k-Group

## 题目描述（改写版）

给你一条单链表的头节点 `head`，以及一个正整数 `k`。请把链表**每 k 个节点一组**进行翻转：

- 一组之内的节点顺序反转；
- 组与组之间的相对顺序保持不变；
- 如果最后剩下的节点不足 k 个，就保持原样，不翻转；
- 只能改指针，不能改节点的值。

## 输入

- `head`：单链表头节点，节点个数 1 到 5000。
- `k`：1 到节点个数。

## 输出

返回翻转后的链表头节点。

## 示例

示例 1：`1 -> 2 -> 3 -> 4 -> 5`，`k = 2`，结果是 `2 -> 1 -> 4 -> 3 -> 5`。

示例 2：`1 -> 2 -> 3 -> 4 -> 5`，`k = 3`，结果是 `3 -> 2 -> 1 -> 4 -> 5`。最后两个节点不足 3 个，保持不动。

示例 3：`1 -> 2 -> 3 -> 4 -> 5`，`k = 1`，结果不变。

## 易漏细节

- 先“数够 k 个”再翻转。数不够就直接结束，剩余部分保持原样。
- 头节点会变，必须用 `dummy`。
- 每段翻转完，要把前一段的尾巴接到本段的新头；同时记下本段的新尾巴，供下一轮使用。


## 解法名称

**分组迭代翻转（Iterative Group Reversal）**。

## 暴力思路

把节点值全部读进列表，按每 k 个一组在列表里反转，再写回节点。改的是值，不符合题目要求。

## 优化思路

四步循环，直到剩余节点不足 k 个：

1. **数**：从 `group_prev` 出发往前走 k 步，找到 `kth`。中途遇到 `None` 说明不够 k 个，直接返回 `dummy.next`。
2. **记**：`group_next = kth.next`，也就是下一段的开头。
3. **翻**：把 `group_prev.next` 到 `kth` 这一段反转，`prev` 从 `group_next` 出发。
4. **接**：`old_start = group_prev.next`（反转后它是本段尾巴），`group_prev.next = kth`，然后 `group_prev = old_start`，进入下一轮。

时间 O(n)，每个节点只被访问常数次；空间 O(1)。

## 画图跟踪

以 `1 -> 2 -> 3 -> 4 -> 5`，`k = 2` 为例：

| 轮次 | group_prev | 本段 | kth | 反转后 |
| --- | --- | --- | --- | --- |
| 第 1 轮 | dummy | 1, 2 | 2 | dummy -> 2 -> 1 -> 3 -> 4 -> 5，group_prev 变成 1 |
| 第 2 轮 | 1 | 3, 4 | 4 | dummy -> 2 -> 1 -> 4 -> 3 -> 5，group_prev 变成 3 |
| 第 3 轮 | 3 | 只剩 5，不够 2 个 | | 结束 |


## 你来写：K 个一组翻转

要求：

- 用 `dummy` + 四步循环写。
- 数够 k 个再翻转，不够就停。
- 写完用 `k = 1`、`k = 2`、`k = 3`、链表长度正好是 k 的倍数、只剩 1 个节点这几种情况各跑一遍。

先在心里回答：为什么 `prev` 要从 `group_next` 出发，而不是从 `None` 出发？


In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


def build_list(values):
    """根据普通列表从头创建一条无环单链表。"""
    dummy = ListNode()
    tail = dummy
    for value in values:
        tail.next = ListNode(value)
        tail = tail.next
    return dummy.next


def list_values(head, limit=30):
    """把无环链表转成普通列表，方便观察结果。"""
    values = []
    current = head
    while current is not None and len(values) < limit:
        values.append(current.val)
        current = current.next
    return values


# 题目：K 个一组翻转链表
# 解法：分组迭代翻转（Iterative Group Reversal）
# 输入：单链表头节点 head（节点数 1 到 5000），正整数 k（1 到节点数）。
# 目标：每 k 个节点一组翻转，不足 k 个的尾部保持原样。
# 输出：返回翻转后的头节点。
# 注意：头节点会变，需要 dummy；先数够 k 个再翻转；反转时 prev 从 group_next 出发；每轮结束 group_prev 移到本段原来的第一个节点。


def reverse_k_group(head, k):
    # 在这里写你的代码
    pass


result = reverse_k_group(build_list([1, 2, 3, 4, 5]), 2)
print(list_values(result))


In [ ]:
print(list_values(reverse_k_group(build_list([1, 2, 3, 4, 5]), 2)))     # 期望 [2, 1, 4, 3, 5]
print(list_values(reverse_k_group(build_list([1, 2, 3, 4, 5]), 3)))     # 期望 [3, 2, 1, 4, 5]
print(list_values(reverse_k_group(build_list([1, 2, 3, 4, 5]), 1)))     # 期望 [1, 2, 3, 4, 5]
print(list_values(reverse_k_group(build_list([1, 2, 3, 4, 5, 6]), 3)))  # 期望 [3, 2, 1, 6, 5, 4]
print(list_values(reverse_k_group(build_list([1]), 1)))                 # 期望 [1]
print(list_values(reverse_k_group(build_list([1, 2]), 3)))              # 期望 [1, 2]


## 参考答案：K 个一组翻转

```python
def reverse_k_group_answer(head, k):
    dummy = ListNode()
    dummy.next = head
    group_prev = dummy

    while True:
        # 第一步：数够 k 个
        kth = group_prev
        for i in range(k):
            kth = kth.next
            if kth is None:
                return dummy.next

        # 第二步：记下下一段的开头
        group_next = kth.next

        # 第三步：反转本段，prev 从 group_next 出发
        prev = group_next
        current = group_prev.next
        while current is not group_next:
            nxt = current.next
            current.next = prev
            prev = current
            current = nxt

        # 第四步：接回前驱，并把 group_prev 移到本段的新尾巴
        old_start = group_prev.next
        group_prev.next = kth
        group_prev = old_start
```

面试表达：

我用虚拟头节点加分组迭代。每一轮先向后走 k 步找到本段的第 k 个节点，如果中途遇到空说明不够 k 个，直接返回结果。够的话记下第 k 个节点的下一个节点作为下一段的开头，然后把这一段反转——反转时让 `prev` 从下一段的开头出发，这样反转完本段原来的第一个节点自然就指向了下一段。最后把前驱指向本段的新头，并把 `group_prev` 移到本段的新尾巴上。时间 O(n)，空间 O(1)。


In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


def build_list(values):
    """根据普通列表从头创建一条无环单链表。"""
    dummy = ListNode()
    tail = dummy
    for value in values:
        tail.next = ListNode(value)
        tail = tail.next
    return dummy.next


def list_values(head, limit=30):
    """把无环链表转成普通列表，方便观察结果。"""
    values = []
    current = head
    while current is not None and len(values) < limit:
        values.append(current.val)
        current = current.next
    return values


def reverse_k_group_answer(head, k):
    dummy = ListNode()
    dummy.next = head
    group_prev = dummy

    while True:
        kth = group_prev
        for i in range(k):
            kth = kth.next
            if kth is None:
                return dummy.next

        group_next = kth.next

        prev = group_next
        current = group_prev.next
        while current is not group_next:
            nxt = current.next
            current.next = prev
            prev = current
            current = nxt

        old_start = group_prev.next
        group_prev.next = kth
        group_prev = old_start


print(list_values(reverse_k_group_answer(build_list([1, 2, 3, 4, 5]), 2)))     # [2, 1, 4, 3, 5]
print(list_values(reverse_k_group_answer(build_list([1, 2, 3, 4, 5]), 3)))     # [3, 2, 1, 4, 5]
print(list_values(reverse_k_group_answer(build_list([1, 2, 3, 4, 5]), 1)))     # [1, 2, 3, 4, 5]
print(list_values(reverse_k_group_answer(build_list([1, 2, 3, 4, 5, 6]), 3)))  # [3, 2, 1, 6, 5, 4]
print(list_values(reverse_k_group_answer(build_list([1]), 1)))                 # [1]


## 题目 4 做题前先补：节点不能直接进堆

合并多条有序链表时，最自然的想法是：把每条链表的当前头节点放进堆，谁小就取谁。

但有个坑：**堆在比较元素大小时，如果两个节点的 `val` 相同，它会试图比较 `ListNode` 对象本身，而对象之间没有大小关系，程序会报错。**

解决办法是给每个节点配一个“不会重复的编号”，组成三元组 `(val, index, node)`：

- 先比 `val`，这是我们要的排序依据；
- `val` 相同时比 `index`，一定分得出先后，永远不会比到 `node`。

这个技巧叫“给堆加裁判”，面试里很常见。


In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


def build_list(values):
    """根据普通列表从头创建一条无环单链表。"""
    dummy = ListNode()
    tail = dummy
    for value in values:
        tail.next = ListNode(value)
        tail = tail.next
    return dummy.next


def list_values(head, limit=30):
    """把无环链表转成普通列表，方便观察结果。"""
    values = []
    current = head
    while current is not None and len(values) < limit:
        values.append(current.val)
        current = current.next
    return values


import heapq

# 演示：为什么第二个元素必须能比较
a = ListNode(1)
b = ListNode(1)
try:
    bad_heap = []
    heapq.heappush(bad_heap, (a.val, a))
    heapq.heappush(bad_heap, (b.val, b))
    print("没有报错")
except TypeError as error:
    print("值相同时，堆会去比较节点对象，报错了：", error)

heap = []
heapq.heappush(heap, (a.val, 0, a))
heapq.heappush(heap, (b.val, 1, b))
print("加上编号就正常：", heap[0][0], heap[0][1])


# 题目 4：合并 K 个升序链表

LeetCode 23. Merge k Sorted Lists

## 题目描述（改写版）

给你一个数组 `lists`，里面有 k 条链表，每条都已经按升序排好。请把它们合并成一条升序链表，返回合并后的头节点。

## 输入

- `lists`：链表数组，k 在 0 到 10000 之间，所有链表的节点总数不超过 10000。

## 输出

返回合并后的链表头节点；如果所有链表都是空的，返回 `None`。

## 示例

示例 1：`lists = [[1,4,5], [1,3,4], [2,6]]`，结果是 `1 -> 1 -> 2 -> 3 -> 4 -> 4 -> 5 -> 6`。

示例 2：`lists = []`，返回空链表。

示例 3：`lists = [[]]`，返回空链表。

## 易漏细节

- 空的链表要先跳过，不能直接放进堆。
- 每次从堆里取出最小的节点，接在结果后面，再把它所在链表的下一个节点放进堆。
- 堆里放三元组 `(值, 链表编号, 节点)`，编号用来避免比较节点对象。


## 解法名称

**小顶堆多路归并（Min-Heap K-Way Merge）**。

## 暴力思路

两两合并：先合并第 1、2 条，再拿结果合并第 3 条……一共合并 k-1 次，每次要遍历已合并的全部节点，总时间 O(k²n)。

## 优化思路

用一个小顶堆，每次从 k 条链表的“当前头节点”里挑最小的：

1. 初始化：把每条非空链表的头节点以 `(值, 编号, 节点)` 放进堆。
2. 循环：弹出堆顶（当前最小的节点），接到结果链表尾部；如果它还有下一个节点，把下一个节点也放进堆。
3. 堆空了就结束。

堆的大小最多是 k，所以每次操作 O(log k)，每个节点进堆出堆各一次，总时间 O(N log k)（N 是节点总数）；空间 O(k)。

另一种常见做法是分治两两合并，时间也是 O(N log k)，可以提一下作为对比。


## 你来写：合并 K 个升序链表

要求：

- 用 `(值, 链表编号, 节点)` 三元组 + 小顶堆写。
- 空链表要跳过，空数组要返回 `None`。
- 写完用示例里的三组数据各跑一遍。

先在心里回答：三元组里的“链表编号”是干什么用的？


In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


def build_list(values):
    """根据普通列表从头创建一条无环单链表。"""
    dummy = ListNode()
    tail = dummy
    for value in values:
        tail.next = ListNode(value)
        tail = tail.next
    return dummy.next


def list_values(head, limit=30):
    """把无环链表转成普通列表，方便观察结果。"""
    values = []
    current = head
    while current is not None and len(values) < limit:
        values.append(current.val)
        current = current.next
    return values


import heapq

# 题目：合并 K 个升序链表
# 解法：小顶堆多路归并（Min-Heap K-Way Merge）
# 输入：链表数组 lists，最长 10000 条，节点总数不超过 10000，每条链表已按升序排列。
# 目标：把所有链表合并成一条升序链表。
# 输出：返回合并后的头节点；如果所有链表都为空，返回 None。
# 注意：空链表要跳过；堆里放 (值, 链表编号, 节点) 三元组，编号用来避免比较节点对象。


def merge_k_lists(lists):
    # 在这里写你的代码
    pass


lists = [build_list([1, 4, 5]), build_list([1, 3, 4]), build_list([2, 6])]
print(list_values(merge_k_lists(lists)))


In [ ]:
lists = [build_list([1, 4, 5]), build_list([1, 3, 4]), build_list([2, 6])]
print(list_values(merge_k_lists(lists)))          # 期望 [1, 1, 2, 3, 4, 4, 5, 6]
print(list_values(merge_k_lists([])))             # 期望 []
print(list_values(merge_k_lists([build_list([])])))   # 期望 []
print(list_values(merge_k_lists([build_list([]), build_list([1])])))   # 期望 [1]
print(list_values(merge_k_lists([build_list([2]), build_list([1]), build_list([3])])))   # 期望 [1, 2, 3]


## 参考答案：合并 K 个升序链表

```python
import heapq


def merge_k_lists_answer(lists):
    heap = []
    for i in range(len(lists)):
        if lists[i] is not None:
            heapq.heappush(heap, (lists[i].val, i, lists[i]))

    dummy = ListNode()
    tail = dummy
    while heap:
        value, index, node = heapq.heappop(heap)
        tail.next = node
        tail = tail.next
        if node.next is not None:
            heapq.heappush(heap, (node.next.val, index, node.next))

    return dummy.next
```

面试表达：

我用小顶堆做多路归并。先把每条非空链表的头节点以 `(值, 链表编号, 节点)` 的形式放进堆，编号是为了防止取值相同时去比较节点对象。然后不断弹出堆顶这个最小节点，接到结果链表后面；如果它还有后继，就把后继也放进堆。堆的大小不超过链表条数 k，所以时间是 O(N log k)，空间 O(k)。也可以用分治两两合并，复杂度一样。


In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


def build_list(values):
    """根据普通列表从头创建一条无环单链表。"""
    dummy = ListNode()
    tail = dummy
    for value in values:
        tail.next = ListNode(value)
        tail = tail.next
    return dummy.next


def list_values(head, limit=30):
    """把无环链表转成普通列表，方便观察结果。"""
    values = []
    current = head
    while current is not None and len(values) < limit:
        values.append(current.val)
        current = current.next
    return values


import heapq


def merge_k_lists_answer(lists):
    heap = []
    for i in range(len(lists)):
        if lists[i] is not None:
            heapq.heappush(heap, (lists[i].val, i, lists[i]))

    dummy = ListNode()
    tail = dummy
    while heap:
        value, index, node = heapq.heappop(heap)
        tail.next = node
        tail = tail.next
        if node.next is not None:
            heapq.heappush(heap, (node.next.val, index, node.next))

    return dummy.next


lists = [build_list([1, 4, 5]), build_list([1, 3, 4]), build_list([2, 6])]
print(list_values(merge_k_lists_answer(lists)))       # [1, 1, 2, 3, 4, 4, 5, 6]
print(list_values(merge_k_lists_answer([])))          # []
print(list_values(merge_k_lists_answer([build_list([])])))   # []
print(list_values(merge_k_lists_answer([build_list([2]), build_list([1]), build_list([3])])))   # [1, 2, 3]


# 今日小结

今天记住三件事：

1. **堆就是“随时拿到最小值”的容器**：`heappush` 进、`heappop` 弹最小、`heap[0]` 看堆顶。
2. **求前 k 大用小顶堆，求前 k 小用大顶堆**（Python 用负数模拟大顶堆）。
3. **堆里放元组时，想按什么排序就把什么放第一位**；如果后面的元素无法比较，就加一个不会重复的编号。

链表部分：

4. **分组翻转**先数够 k 个，再反转；`prev` 从下一段的开头出发，反转完自然接上。
5. **合并 K 条链表**用小顶堆多路归并，每次弹出最小的那个节点。


## 今日复盘区

- `heapq` 的 `heappush`、`heappop`、`heap[0]` 分别做什么？
- 为什么求第 k 大要维护小顶堆？如果维护大顶堆会怎样？
- 分组翻转里，`group_prev` 每轮要移到哪里？
- 合并 K 条链表时，三元组里的编号不能省，为什么？
- 今天的四道题里，哪一道能不看答案写出来？

完成情况记录：

- 独立写出：
- 卡住的题：
- 明天重写：
- 完成日期：
